# NumPy: From Basics to Expert 🧮

**A hands-on, teaching-first notebook.**

NumPy (Numerical Python) is the foundation of the entire scientific Python stack — pandas, scikit-learn, SciPy, TensorFlow, PyTorch, OpenCV, and matplotlib all build on it. If you understand NumPy deeply, everything else gets easier.

### How to use this notebook
- **Read the markdown, then run the code cell below it.** Don't just read — run, tweak a number, run again.
- Each section ends with **🎯 Try it yourself** exercises. Attempt them before peeking.
- Cells build on each other, so run them top to bottom (use *Run All* the first time).

### Roadmap
| Part | Topic | Level |
|------|-------|-------|
| 1 | Why NumPy? Lists vs arrays | Basic |
| 2 | Creating arrays | Basic |
| 3 | Array attributes (shape, dtype, ndim) | Basic |
| 4 | Indexing & slicing | Basic |
| 5 | Boolean & fancy indexing | Intermediate |
| 6 | Reshaping & combining arrays | Intermediate |
| 7 | Vectorized math & ufuncs | Intermediate |
| 8 | Broadcasting (the big one) | Intermediate |
| 9 | Aggregations & the `axis` argument | Intermediate |
| 10 | Random numbers (modern API) | Intermediate |
| 11 | Linear algebra | Advanced |
| 12 | Views vs copies & memory model | Advanced |
| 13 | Strides, `einsum`, structured arrays | Expert |
| 14 | Performance: vectorization & pitfalls | Expert |
| 15 | Capstone mini-project | Expert |
| 16 | Bonus: animations 🎬 | Expert |
| 17 | Deep dive: FFT / signal processing 🌊 | Expert |
| 18 | Deep dive: masked arrays (`np.ma`) 🩹 | Expert |
| 19 | Deep dive: memory-mapped files 💾 | Expert |

📊 **Visualizations included** — throughout the notebook we use **matplotlib** to *see* what the arrays are doing (broadcasting grids, distributions, Monte Carlo, heatmaps...). Run the setup cell below once.

Let's go. 🚀


In [ ]:
# Visualization setup — run this once.
import numpy as np
import matplotlib.pyplot as plt

# Show plots inline inside the notebook.
%matplotlib inline

# A clean, consistent look for every figure in this notebook.
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["figure.dpi"] = 100

print("matplotlib ready:", plt.matplotlib.__version__)


## Part 1 — Why NumPy? Python lists vs NumPy arrays

A Python list can hold anything, but it's slow for math: each element is a separate Python object, and looping happens in the (slow) Python interpreter.

A **NumPy array** (`ndarray`) stores numbers in one contiguous block of memory, all of the same type. Operations run in compiled C code over the whole array at once — this is called **vectorization**.

Two consequences:
1. **Speed**: often 10–100× faster than pure Python loops.
2. **Expressiveness**: `a + b` adds two whole arrays — no loop needed.


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)

# A Python list "multiply" repeats; a NumPy array does elementwise math.
py_list = [1, 2, 3, 4]
np_arr  = np.array([1, 2, 3, 4])

print("list * 2 :", py_list * 2)     # [1, 2, 3, 4, 1, 2, 3, 4]  (repetition!)
print("array * 2:", np_arr * 2)      # [2 4 6 8]                 (math!)


In [ ]:
# Speed demo: sum of squares for a million numbers.
import time

N = 1_000_000
py = list(range(N))
np_arr = np.arange(N)

t0 = time.perf_counter()
s1 = sum(x * x for x in py)
t1 = time.perf_counter()

s2 = np.sum(np_arr * np_arr)         # vectorized
t2 = time.perf_counter()

print(f"Pure Python : {t1 - t0:.4f} s")
print(f"NumPy       : {t2 - t1:.4f} s")
print(f"Speedup     : {(t1 - t0) / (t2 - t1):.1f}x   (same answer: {s1 == s2})")


**Takeaway:** when you find yourself writing a `for` loop over numbers, ask *"can NumPy do this on the whole array at once?"* The answer is usually yes.

### 🎯 Try it yourself
1. Create a list `[10, 20, 30]` and a NumPy array of the same values. Add `5` to each — notice the list errors but the array works.
2. Time how long `np.sum(np_arr)` takes vs `sum(py)`.


## Part 2 — Creating arrays

There are many ways to make arrays. The most common:

| Function | What it makes |
|----------|---------------|
| `np.array([...])` | from a Python list/tuple |
| `np.zeros(shape)` / `np.ones(shape)` | filled with 0 / 1 |
| `np.full(shape, value)` | filled with a constant |
| `np.arange(start, stop, step)` | like `range`, but an array |
| `np.linspace(start, stop, num)` | `num` evenly spaced points (endpoint included) |
| `np.eye(n)` | identity matrix |
| `np.empty(shape)` | uninitialized (fast, contains garbage) |


In [ ]:
print("array     :", np.array([1, 2, 3]))
print("zeros(4)  :", np.zeros(4))
print("ones(3,2) :\n", np.ones((3, 2)))
print("full      :", np.full(3, 7))
print("arange    :", np.arange(0, 10, 2))        # 0 2 4 6 8 (stop excluded)
print("linspace  :", np.linspace(0, 1, 5))       # 0 .25 .5 .75 1 (stop included)
print("eye(3)    :\n", np.eye(3))


**`arange` vs `linspace`** — a classic source of confusion:
- `arange` — you specify the **step**; the endpoint is **excluded**. Good for integer sequences.
- `linspace` — you specify the **count**; the endpoint is **included**. Good for plotting / sampling a range.

Prefer `linspace` for floats: `arange(0, 1, 0.1)` can give surprising rounding (you might get 11 elements or a near-`1.0` last value).


In [ ]:
# Note the float surprise:
print("arange floats  :", np.arange(0, 1, 0.1), "-> len", len(np.arange(0, 1, 0.1)))
print("linspace clean :", np.linspace(0, 1, 11))


### 📊 Visualize: `linspace` is how you sample a curve

`linspace` gives you a smooth set of x-values to feed into a function and plot. This is the single most common plotting pattern in scientific Python.


In [ ]:
x = np.linspace(0, 2 * np.pi, 200)     # 200 smooth points from 0 to 2π

plt.figure()
plt.plot(x, np.sin(x), label="sin(x)")
plt.plot(x, np.cos(x), label="cos(x)", linestyle="--")
plt.title("np.linspace + ufuncs = smooth curves")
plt.xlabel("x"); plt.ylabel("value")
plt.legend()
plt.show()

# Fewer points -> you can SEE the sampling (markers at each linspace value):
xs = np.linspace(0, 2 * np.pi, 15)
plt.figure()
plt.plot(x, np.sin(x), color="lightgray", label="true curve")
plt.plot(xs, np.sin(xs), "o-", label="15 linspace samples")
plt.title("Coarse vs fine sampling")
plt.legend()
plt.show()


### 🎯 Try it yourself
1. Make a 4×4 array of all 7s.
2. Make 9 evenly spaced numbers from -1 to 1 with `linspace`.
3. Make `[1, 3, 5, 7, 9]` two different ways (`arange` and `linspace`).
4. Plot `y = x**2` for `x = np.linspace(-3, 3, 100)`.


## Part 3 — Array attributes: shape, dtype, ndim, size

Every array carries metadata you'll inspect constantly:
- **`.shape`** — a tuple of dimension sizes, e.g. `(3, 4)` = 3 rows, 4 columns.
- **`.ndim`** — number of dimensions (axes).
- **`.size`** — total number of elements.
- **`.dtype`** — the element type (`int64`, `float64`, `bool`, ...).
- **`.itemsize`** / **`.nbytes`** — bytes per element / total bytes.

> **Axis intuition:** axis 0 runs **down the rows**, axis 1 runs **across the columns**. For 3D `(depth, rows, cols)`, axis 0 is depth. This matters hugely for aggregations (Part 9).


In [ ]:
a = np.array([[1, 2, 3],
              [4, 5, 6]])

print("array:\n", a)
print("shape    :", a.shape)     # (2, 3)
print("ndim     :", a.ndim)      # 2
print("size     :", a.size)      # 6
print("dtype    :", a.dtype)     # int64 (or int32 on some Windows builds)
print("itemsize :", a.itemsize, "bytes/element")
print("nbytes   :", a.nbytes, "bytes total")


### Controlling and changing the dtype

The dtype is chosen automatically, but you can set it explicitly or convert later with `.astype()`. Choosing a smaller dtype (e.g. `float32`, `int8`) saves memory — critical for big data and deep learning.


In [ ]:
f = np.array([1, 2, 3], dtype=np.float32)
print(f, f.dtype)

g = f.astype(np.int16)           # convert (makes a copy)
print(g, g.dtype)

# Beware integer overflow with small dtypes!
small = np.array([250], dtype=np.uint8)   # uint8 holds 0..255
print("250 + 10 as uint8 =", small + np.uint8(10))   # wraps around to 4!


**⚠️ Overflow** is silent in NumPy. `250 + 10` in `uint8` wraps to `4` because the type only holds 0–255. When in doubt, use a wider dtype.

### 🎯 Try it yourself
1. Build a 3D array of shape `(2, 3, 4)` with `np.zeros` and print its `ndim` and `size`.
2. Create `np.array([1.7, 2.9, 3.1])` and convert to int — does it round or truncate?


## Part 4 — Indexing & slicing

Indexing is like Python lists, extended to multiple dimensions.

- 1D: `a[i]`, `a[start:stop:step]`
- 2D: `a[row, col]`, `a[row_slice, col_slice]`
- Negative indices count from the end: `a[-1]` is the last element.
- A slice is `start:stop:step`; any part can be omitted (`a[:, 1]` = all rows, column 1).

> **Key fact:** a basic **slice returns a *view*** — it shares memory with the original. Editing the slice edits the original! (More in Part 12.)


In [ ]:
a = np.arange(1, 13).reshape(3, 4)
print("a =\n", a)
print()
print("a[0, 0]    =", a[0, 0])        # first element -> 1
print("a[-1, -1]  =", a[-1, -1])      # last element  -> 12
print("a[1]       =", a[1])           # whole row 1   -> [5 6 7 8]
print("a[:, 2]    =", a[:, 2])        # column 2      -> [3 7 11]
print("a[0:2, 1:3]=\n", a[0:2, 1:3])  # sub-block
print("every other column a[:, ::2] =\n", a[:, ::2])


In [ ]:
# Slices are VIEWS: this edits the original!
a = np.arange(1, 13).reshape(3, 4)
row = a[0]            # a view of the first row
row[:] = 0           # write into the view
print("after editing the view, original a:\n", a)   # first row is now zeros


### 🎯 Try it yourself
Given `a = np.arange(1, 26).reshape(5, 5)`:
1. Extract the center 3×3 block.
2. Reverse the rows (top becomes bottom). *(Hint: `a[::-1]`.)*
3. Set the last column to `-1` without a loop.


## Part 5 — Boolean & fancy indexing

This is where NumPy starts to feel magical and you stop writing loops.

### Boolean masking
A comparison like `a > 5` produces a **boolean array**. Indexing with it keeps only the `True` elements.

### Fancy (integer-array) indexing
Index with a **list/array of integer positions** to grab elements in any order, with repeats.

Unlike slices, **both of these return copies, not views.**


In [ ]:
a = np.array([12, 5, 8, 21, 3, 17, 9])

mask = a > 10
print("mask      :", mask)            # [ True False ... ]
print("a[a > 10] :", a[mask])         # [12 21 17]

# Modify in place via a mask — clip everything below 10 up to 10:
b = a.copy()
b[b < 10] = 10
print("clipped   :", b)

# Combine conditions with & (and), | (or), ~ (not). PARENTHESES ARE REQUIRED.
print("10<a<20   :", a[(a > 10) & (a < 20)])


In [ ]:
# Fancy indexing: pick positions in any order, with repeats.
a = np.array([100, 101, 102, 103, 104])
idx = [4, 0, 0, 2]
print("a[idx]    :", a[idx])          # [104 100 100 102]

# In 2D, pass row indices and column indices together:
m = np.arange(1, 13).reshape(3, 4)
rows = [0, 1, 2]
cols = [0, 2, 3]
print("diagonal-ish picks:", m[rows, cols])   # m[0,0], m[1,2], m[2,3] -> [1 7 12]


**⚠️ Gotcha:** use `&`, `|`, `~` (bitwise) with parentheses — **not** Python's `and`/`or`/`not`, which fail on arrays. `(a > 10) & (a < 20)`, never `a > 10 and a < 20`.

Useful companions:
- `np.where(cond, x, y)` — vectorized if/else.
- `np.nonzero(a)` / `np.argwhere(a)` — positions where condition holds.
- `np.any(mask)` / `np.all(mask)` — reduce a mask to one bool.


In [ ]:
a = np.array([12, 5, 8, 21, 3, 17, 9])
print("np.where label :", np.where(a > 10, 1, 0))    # 1 where big, else 0
print("positions >10  :", np.nonzero(a > 10)[0])     # indices: [0 3 5]
print("any >20?       :", np.any(a > 20))
print("all >0?        :", np.all(a > 0))


### 🎯 Try it yourself
Given `data = np.array([3, -1, 4, -1, 5, 9, -2, 6])`:
1. Replace every negative value with `0` using a mask.
2. Use `np.where` to make an array that is `"hi"` where `data > 4` else `"lo"`.
3. Count how many values are even. *(Hint: `data % 2 == 0`, then `.sum()`.)*


## Part 6 — Reshaping & combining arrays

The **same data** can be viewed in different shapes as long as the total element count matches.

- **`.reshape(...)`** — change shape; use `-1` to let NumPy infer one dimension.
- **`.ravel()` / `.flatten()`** — make 1D (`ravel` is a view when possible; `flatten` always copies).
- **`.T`** / **`np.transpose`** — swap axes.
- **`np.newaxis`** / `.reshape` — add a length-1 axis (turn a row into a column).
- **Stacking:** `np.concatenate`, `np.vstack`, `np.hstack`, `np.stack`.
- **Splitting:** `np.split`, `np.hsplit`, `np.vsplit`.


In [ ]:
a = np.arange(12)
print("original :", a)
print("reshape 3x4:\n", a.reshape(3, 4))
print("reshape 2x-1 (infer):\n", a.reshape(2, -1))   # -1 -> 6 columns

m = a.reshape(3, 4)
print("transpose:\n", m.T)
print("ravel    :", m.ravel())


In [ ]:
# Row vector -> column vector with newaxis
v = np.array([1, 2, 3])
print("v shape       :", v.shape)
col = v[:, np.newaxis]            # same as v.reshape(3, 1)
print("column shape  :", col.shape)
print(col)


In [ ]:
# Combining arrays
x = np.array([[1, 2],
              [3, 4]])
y = np.array([[5, 6],
              [7, 8]])

print("vstack (stack rows):\n", np.vstack([x, y]))
print("hstack (side by side):\n", np.hstack([x, y]))
print("concatenate axis=0:\n", np.concatenate([x, y], axis=0))
print("stack -> new axis, shape:", np.stack([x, y]).shape)   # (2, 2, 2)


### 🎯 Try it yourself
1. Turn `np.arange(24)` into shape `(2, 3, 4)`, then flatten it back.
2. Stack `np.arange(3)` and `np.arange(3, 6)` into a 2×3 array, then transpose to 3×2.
3. Split `np.arange(10)` into 5 equal pieces with `np.split`.


## Part 7 — Vectorized math & universal functions (ufuncs)

A **ufunc** operates elementwise over arrays in compiled C. Arithmetic operators (`+ - * / ** %`) are ufuncs, and so are `np.sqrt`, `np.exp`, `np.sin`, `np.log`, `np.maximum`, and many more.

The rule: **stop looping, operate on whole arrays.**


In [ ]:
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])

print("a + b   :", a + b)
print("b / a   :", b / a)
print("a ** 2  :", a ** 2)
print("sqrt(b) :", np.sqrt(b))
print("exp(a)  :", np.exp(a))
print("max pair:", np.maximum(a, [4, 1, 5, 2]))   # elementwise max


In [ ]:
# Real example: convert Celsius to Fahrenheit for a whole array — no loop.
celsius = np.array([0, 10, 20, 30, 37, 100])
fahrenheit = celsius * 9 / 5 + 32
print("C:", celsius)
print("F:", fahrenheit)

# Vectorized "safe" operations: np.where avoids divide-by-zero warnings.
x = np.array([1.0, 0.0, 2.0, 0.0])
recip = np.where(x != 0, 1 / np.where(x == 0, 1, x), 0.0)
print("safe reciprocal:", recip)


**Out parameter & in-place:** ufuncs accept `out=` to write results into an existing array (avoids allocation), and `+=`, `*=` work in place.

```python
np.multiply(a, 2, out=a)   # a is modified in place
a *= 2                     # same idea
```

### 🎯 Try it yourself
1. Given `prices = np.array([19.99, 5.49, 99.0])`, apply a 15% discount in one line.
2. Compute the elementwise hypotenuse of `legs_a=[3,5,8]` and `legs_b=[4,12,15]`. *(Try `np.hypot`.)*


## Part 8 — Broadcasting ⭐ (the most important idea)

**Broadcasting** lets NumPy operate on arrays of *different shapes* without copying data, by virtually "stretching" the smaller one.

### The rules
Compare shapes **right to left**. Two dimensions are compatible when:
1. they are **equal**, or
2. one of them is **1** (it gets stretched), or
3. one is **missing** (treated as 1).

If any pair fails, you get a `ValueError`.

Examples:
```
(3, 4) + (4,)      -> (3, 4)     # the (4,) row is added to every row
(3, 1) + (1, 4)    -> (3, 4)     # outer-product-like grid
(3, 4) + (3, 1)    -> (3, 4)     # the (3,1) column added to every column
(3, 4) + (3,)      -> ERROR      # 4 vs 3 mismatch
```


In [ ]:
# Add a per-column offset to every row:
m = np.arange(12).reshape(3, 4)
col_offset = np.array([100, 200, 300, 400])     # shape (4,)
print("m + row vector:\n", m + col_offset)

# Add a per-row offset to every column (needs a column shape):
row_offset = np.array([10, 20, 30]).reshape(3, 1)   # shape (3,1)
print("\nm + column vector:\n", m + row_offset)


In [ ]:
# Classic use: build a multiplication table via broadcasting (no loops).
r = np.arange(1, 6).reshape(5, 1)   # (5,1) column
c = np.arange(1, 6).reshape(1, 5)   # (1,5) row
print("Multiplication table:\n", r * c)

# Standardize each column of a data matrix (z-score):
data = np.array([[1., 2., 3.],
                 [4., 5., 6.],
                 [7., 8., 9.]])
mean = data.mean(axis=0)            # shape (3,)
std  = data.std(axis=0)            # shape (3,)
z = (data - mean) / std            # broadcasts (3,3)-(3,)
print("\nColumn-standardized:\n", z.round(3))


### 📊 Visualize: broadcasting builds 2D fields from 1D vectors

A `(N,1)` column times a `(1,M)` row "fills in" a whole grid. This is exactly how you evaluate a function `f(x, y)` over a 2D plane. We'll show the multiplication table as a heatmap, then a real 2D function.


In [ ]:
# 1) Multiplication table as a heatmap
r = np.arange(1, 11).reshape(10, 1)
c = np.arange(1, 11).reshape(1, 10)
table = r * c

plt.figure(figsize=(5.5, 5))
plt.imshow(table, cmap="viridis", origin="lower",
           extent=[0.5, 10.5, 0.5, 10.5])
plt.colorbar(label="product")
# annotate each cell
for i in range(10):
    for j in range(10):
        plt.text(j + 1, i + 1, table[i, j], ha="center", va="center",
                 color="white", fontsize=7)
plt.title("Broadcasting: (10,1) × (1,10) → 10×10 grid")
plt.xlabel("column value"); plt.ylabel("row value")
plt.show()


In [ ]:
# 2) Evaluate f(x, y) = sin(x) * cos(y) over a plane — pure broadcasting, no loops.
x = np.linspace(-3, 3, 300).reshape(1, -1)   # row    (1, 300)
y = np.linspace(-3, 3, 300).reshape(-1, 1)   # column (300, 1)
field = np.sin(x) * np.cos(y)                # broadcasts to (300, 300)

plt.figure(figsize=(6, 5))
plt.imshow(field, cmap="RdBu", origin="lower", extent=[-3, 3, -3, 3])
plt.colorbar(label="sin(x)·cos(y)")
plt.title("A 2D field from two 1D arrays (broadcasting)")
plt.xlabel("x"); plt.ylabel("y")
plt.show()
print("field shape:", field.shape, "— built with NO explicit loops")


**Why it matters:** broadcasting is how you normalize data, add bias vectors in neural nets, compute pairwise distances, and more — all without explicit loops and without wasting memory.

### 🎯 Try it yourself
1. Given a `(4, 3)` array, subtract the **mean of each row** so every row sums to ~0. *(Hint: `mean(axis=1, keepdims=True)`.)*
2. Build a 10×10 grid where entry `[i, j] = i + j` using broadcasting.
3. Why does `np.ones((3, 4)) + np.ones((2, 4))` fail? Predict, then test.


## Part 9 — Aggregations & the `axis` argument

Reductions collapse data: `sum`, `mean`, `min`, `max`, `std`, `var`, `prod`, `argmin`, `argmax`, `cumsum`, ...

The crucial parameter is **`axis`**:
- `axis=None` (default) — reduce the **whole array** to a scalar.
- `axis=0` — collapse **down the rows** → one value **per column**.
- `axis=1` — collapse **across the columns** → one value **per row**.

> Mnemonic: *the axis you name is the axis that disappears.*


In [ ]:
m = np.array([[1, 2, 3],
              [4, 5, 6]])

print("sum all      :", m.sum())            # 21
print("sum axis=0   :", m.sum(axis=0))      # per column -> [5 7 9]
print("sum axis=1   :", m.sum(axis=1))      # per row    -> [6 15]
print("mean axis=0  :", m.mean(axis=0))
print("max axis=1   :", m.max(axis=1))
print("argmax all   :", m.argmax())         # flat index of the maximum
print("cumsum axis=1:\n", m.cumsum(axis=1))


In [ ]:
# keepdims=True preserves the reduced axis as length 1 -> handy for broadcasting back.
m = np.arange(12).reshape(3, 4).astype(float)
row_means = m.mean(axis=1, keepdims=True)   # shape (3,1) instead of (3,)
print("row_means shape:", row_means.shape)
print("rows centered:\n", (m - row_means))  # broadcasts cleanly


### NaN-aware aggregations
Real data has missing values (`np.nan`). Normal `sum`/`mean` return `nan` if any element is `nan`. Use `np.nansum`, `np.nanmean`, `np.nanmax`, ... to skip them.


In [ ]:
vals = np.array([1.0, 2.0, np.nan, 4.0])
print("mean      :", np.mean(vals))      # nan
print("nanmean   :", np.nanmean(vals))   # 2.333...  (ignores nan)


### 📊 Visualize: what `axis` actually collapses

Same data, two reductions. `axis=0` gives a value per **column**; `axis=1` gives a value per **row**. Seeing them side-by-side cements the "the named axis disappears" rule.


In [ ]:
scores = np.array([[80, 90, 70],
                   [60, 75, 95],
                   [100, 50, 85]])
students = ["Ann", "Ben", "Cy"]
exams = ["Math", "Sci", "Hist"]

per_exam    = scores.mean(axis=0)   # collapse rows -> per column (exam)
per_student = scores.mean(axis=1)   # collapse cols -> per row (student)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].bar(exams, per_exam, color="steelblue")
ax[0].set_title("mean(axis=0): per exam"); ax[0].set_ylabel("avg score")
ax[1].bar(students, per_student, color="indianred")
ax[1].set_title("mean(axis=1): per student"); ax[1].set_ylabel("avg score")
plt.tight_layout(); plt.show()


### 🎯 Try it yourself
Given `scores = np.array([[80, 90, 70], [60, 75, 95], [100, 50, 85]])` where rows are students and columns are exams:
1. Each student's average (one number per row).
2. Each exam's highest score (one number per column).
3. Which student had the single best score? *(Hint: `np.unravel_index(scores.argmax(), scores.shape)`.)*


## Part 10 — Random numbers (the modern API)

Modern NumPy uses a **`Generator`** object created by `np.random.default_rng(seed)`. Seeding makes results reproducible — essential for experiments and debugging.

> The old `np.random.rand` / `np.random.seed` functions still work but the `default_rng` API is preferred: it's faster, statistically better, and avoids hidden global state.


In [ ]:
rng = np.random.default_rng(seed=42)   # reproducible

print("uniform [0,1):\n", rng.random((2, 3)))
print("integers 0..9 :", rng.integers(0, 10, size=8))
print("normal mu=0 sd=1:\n", rng.normal(0, 1, size=5).round(3))
print("shuffle pick   :", rng.choice(['a', 'b', 'c', 'd'], size=3, replace=False))


In [ ]:
# Same seed -> same numbers (reproducibility).
a = np.random.default_rng(0).integers(0, 100, 5)
b = np.random.default_rng(0).integers(0, 100, 5)
print("run 1:", a)
print("run 2:", b)
print("identical?", np.array_equal(a, b))

# Monte Carlo estimate of pi: fraction of random points inside the unit circle * 4.
rng = np.random.default_rng(1)
pts = rng.random((1_000_000, 2))
inside = (pts[:, 0]**2 + pts[:, 1]**2) <= 1.0
print("pi estimate:", 4 * inside.mean())


### 📊 Visualize: distributions & the Monte Carlo π estimate

Histograms show the *shape* of random data. The π demo is far more convincing as a picture: points inside the quarter-circle vs outside.


In [ ]:
rng = np.random.default_rng(7)

# Compare two distributions with histograms
uniform = rng.random(100_000)
normal  = rng.normal(0.5, 0.15, 100_000)

plt.figure()
plt.hist(uniform, bins=50, alpha=0.6, label="uniform [0,1)", density=True)
plt.hist(normal,  bins=50, alpha=0.6, label="normal μ=0.5 σ=0.15", density=True)
plt.title("Two distributions, 100k samples each")
plt.xlabel("value"); plt.ylabel("density"); plt.legend()
plt.show()


In [ ]:
# Monte Carlo π, visualized
rng = np.random.default_rng(1)
n = 5000
pts = rng.random((n, 2))
inside = (pts[:, 0]**2 + pts[:, 1]**2) <= 1.0
pi_est = 4 * inside.mean()

plt.figure(figsize=(5.5, 5.5))
plt.scatter(pts[inside, 0],  pts[inside, 1],  s=4, color="seagreen", label="inside")
plt.scatter(pts[~inside, 0], pts[~inside, 1], s=4, color="lightcoral", label="outside")
theta = np.linspace(0, np.pi / 2, 100)
plt.plot(np.cos(theta), np.sin(theta), "k-", lw=2)
plt.gca().set_aspect("equal")
plt.title(f"Monte Carlo π ≈ {pi_est:.4f}  (n={n})")
plt.legend(loc="upper right")
plt.show()


### 🎯 Try it yourself
1. Simulate rolling two dice 10,000 times; estimate the probability the sum is 7, then plot a histogram of the sums.
2. Generate 1,000 samples from a normal distribution with mean 50, std 10; print the sample mean/std and plot a histogram.


## Part 11 — Linear algebra

NumPy's `@` operator and `np.linalg` module cover matrix math.

- `A @ B` — matrix multiplication (use this, not `*`, which is elementwise!).
- `A.T` — transpose. `np.linalg.inv(A)` — inverse. `np.linalg.det(A)` — determinant.
- `np.linalg.solve(A, b)` — solve `Ax = b` (better than computing the inverse).
- `np.linalg.eig`, `np.linalg.svd`, `np.linalg.norm` — eigen-decomposition, SVD, norms.


In [ ]:
A = np.array([[2., 1.],
              [1., 3.]])
B = np.array([[1., 0.],
              [0., 2.]])

print("A @ B (matrix mult):\n", A @ B)
print("A * B (elementwise!):\n", A * B)   # different — be careful
print("transpose:\n", A.T)
print("determinant:", np.linalg.det(A))
print("inverse:\n", np.linalg.inv(A))


In [ ]:
# Solve a linear system:  2x +  y = 5 ;  x + 3y = 10
A = np.array([[2., 1.],
              [1., 3.]])
b = np.array([5., 10.])
x = np.linalg.solve(A, b)
print("solution [x, y]:", x)
print("check A@x == b :", np.allclose(A @ x, b))

# Eigenvalues / norms
vals, vecs = np.linalg.eig(A)
print("eigenvalues:", vals.round(3))
print("L2 norm of b:", np.linalg.norm(b))


**Tip:** prefer `np.linalg.solve(A, b)` over `np.linalg.inv(A) @ b` — it's more accurate and faster.

### 🎯 Try it yourself
1. Multiply a `(2,3)` matrix by a `(3,2)` matrix; what's the result shape?
2. Solve `3x + 2y = 12`, `x - y = 1`.
3. Compute the dot product of `[1,2,3]` and `[4,5,6]` two ways (`@` and `np.dot`).


## Part 12 — Views vs copies & the memory model 🧠

This trips up almost everyone. Understanding it prevents nasty bugs.

- **View** — a new array object that **shares the same data buffer**. Basic slicing (`a[1:3]`), `reshape`, `.T`, and `ravel` (when possible) give views. Editing a view edits the original.
- **Copy** — independent data. Fancy indexing, boolean indexing, `.copy()`, and `astype` give copies.

Use `.base` (a view's `.base` points to the original) and `np.shares_memory(a, b)` to check.


In [ ]:
a = np.arange(10)

view = a[2:5]                 # VIEW (basic slice)
view[0] = 999
print("after editing view, a:", a)         # a[2] became 999
print("shares memory? ", np.shares_memory(a, view))

copy = a[[2, 3, 4]]          # COPY (fancy indexing)
copy[0] = -1
print("after editing copy, a:", a)          # unchanged
print("shares memory? ", np.shares_memory(a, copy))


**Rule of thumb:**
- Need to modify the original through a sub-array? Slice (view).
- Need an independent snapshot? Add `.copy()`.

When unsure whether you have a view, **call `.copy()`** — a tiny cost that buys correctness.

### 🎯 Try it yourself
1. Create `a = np.arange(6).reshape(2,3)`. Take `b = a.T`. Change `b[0,0]` — does `a` change? Why?
2. Take `c = a[a > 2]`, change an element of `c` — does `a` change? Why not?


## Part 13 — Expert tools: strides, `einsum`, structured arrays

### Strides
An ndarray is just a 1D memory block + **strides** (bytes to step to reach the next element along each axis). Reshapes and transposes are cheap because they only change shape/strides, not data. This is why `.T` is instant even for huge arrays.


In [ ]:
a = np.arange(12).reshape(3, 4)
print("shape  :", a.shape)
print("strides:", a.strides, "(bytes to step per axis)")
print("T strides:", a.T.strides, "(just swapped — no data moved)")


### `np.einsum` — Einstein summation
A compact, powerful notation for sums of products: transposes, dot products, matrix multiply, traces, batched ops — all in one function. Each letter labels an axis; repeated letters are summed; letters dropped from the output are summed out.


In [ ]:
A = np.arange(6).reshape(2, 3)
B = np.arange(6).reshape(3, 2)

print("matrix mult  :", "\n", np.einsum('ij,jk->ik', A, B))   # == A @ B
print("trace of sq  :", np.einsum('ii->', np.eye(4) * 5))     # sum of diagonal = 20
print("transpose    :", "\n", np.einsum('ij->ji', A))
print("row sums     :", np.einsum('ij->i', A))                # sum over j
print("batched dot  :", np.einsum('ij,ij->i', A, A))          # per-row self dot


### Structured arrays
Arrays where each element is a small record with named fields — like a lightweight table. (For real tabular work, reach for pandas, but structured arrays are useful for binary formats and interop.)


In [ ]:
people = np.array(
    [('Alice', 30, 5.5), ('Bob', 25, 6.1), ('Cara', 35, 5.9)],
    dtype=[('name', 'U10'), ('age', 'i4'), ('height', 'f4')]
)
print("names :", people['name'])
print("ages  :", people['age'])
print("oldest:", people['name'][people['age'].argmax()])


### 🎯 Try it yourself
1. Use `einsum` to compute the dot product of two 1D vectors `'i,i->'`.
2. Use `einsum` to extract the diagonal of a square matrix (`'ii->i'`).
3. Make a structured array of 3 products with fields `name`, `price`, `qty`; compute total inventory value with broadcasting.


## Part 14 — Performance: think in arrays, avoid the traps

### Golden rules
1. **Vectorize** — replace Python loops with array operations.
2. **Preallocate** — don't grow arrays with `np.append` in a loop (it copies every time). Allocate once with `np.empty`/`np.zeros` and fill.
3. **Avoid unnecessary copies** — reuse buffers with `out=`, prefer views.
4. **Pick the right dtype** — `float32` halves memory vs `float64`.
5. **Use built-in reductions** — `a.sum()` beats `sum(a)`.


In [ ]:
import time

N = 2_000_000
x = np.random.default_rng(0).random(N)

# Slow: Python loop
t0 = time.perf_counter()
total = 0.0
for v in x:
    total += v * v
t1 = time.perf_counter()

# Fast: vectorized
t2 = time.perf_counter()
total_vec = np.sum(x * x)
t3 = time.perf_counter()

print(f"loop       : {t1 - t0:.4f}s")
print(f"vectorized : {t3 - t2:.4f}s")
print(f"speedup    : {(t1 - t0)/(t3 - t2):.0f}x")


In [ ]:
# Preallocation vs growing in a loop
N = 50_000

t0 = time.perf_counter()
grow = np.array([])
for i in range(N):
    grow = np.append(grow, i)          # copies the whole array every iteration!
t1 = time.perf_counter()

t2 = time.perf_counter()
pre = np.empty(N)
for i in range(N):
    pre[i] = i                         # writes into existing memory
t3 = time.perf_counter()

print(f"np.append in loop : {t1 - t0:.4f}s  (quadratic — avoid!)")
print(f"preallocated fill : {t3 - t2:.4f}s")
print("(Best of all: just np.arange(N) — fully vectorized.)")


### 📊 Visualize: vectorization wins (log scale)

Let's time the same computation at growing sizes for a Python loop vs NumPy and plot it. The gap widens dramatically — note the **log scale** on the y-axis.


In [ ]:
sizes = [10_000, 50_000, 100_000, 500_000, 1_000_000]
loop_times, vec_times = [], []

for n in sizes:
    x = np.random.default_rng(0).random(n)

    t0 = time.perf_counter()
    s = 0.0
    for v in x:
        s += v * v
    loop_times.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    _ = np.sum(x * x)
    vec_times.append(time.perf_counter() - t0)

plt.figure()
plt.plot(sizes, loop_times, "o-", label="Python loop")
plt.plot(sizes, vec_times,  "s-", label="NumPy vectorized")
plt.yscale("log")
plt.xlabel("array size"); plt.ylabel("time (s, log scale)")
plt.title("Why we vectorize")
plt.legend()
plt.show()


**⚠️ Common traps**
- Using Python `and`/`or` on arrays → `ValueError`. Use `&`, `|` with parentheses.
- Comparing floats with `==`. Use `np.isclose` / `np.allclose`.
- Forgetting that a slice is a view and accidentally mutating the original.
- Integer overflow with small dtypes (silent wraparound).

### 🎯 Try it yourself
Rewrite this loop as a single vectorized expression:
```python
out = []
for v in data:
    out.append(v**2 if v > 0 else 0)
```
*(Hint: `np.where(data > 0, data**2, 0)`.)*


## Part 15 — Capstone mini-project: analyze a "sales" dataset 📊

Let's tie everything together. We'll simulate a month of sales for several products across several stores, then answer business questions — **using no explicit loops over the data.**

This exercises: random generation, broadcasting, axis-aware aggregation, boolean masking, and `argmax`.


In [ ]:
rng = np.random.default_rng(2024)

products = np.array(["Widget", "Gadget", "Gizmo", "Doohickey"])
stores   = np.array(["North", "South", "East", "West", "Central"])
days = 30

# sales[p, s, d] = units of product p sold at store s on day d
sales = rng.integers(0, 50, size=(len(products), len(stores), days))
prices = np.array([9.99, 19.99, 4.99, 14.99])   # one price per product

print("sales shape (products, stores, days):", sales.shape)
print("total units sold across everything :", sales.sum())


In [ ]:
# 1) Total revenue per product = (units summed over stores & days) * price
units_per_product = sales.sum(axis=(1, 2))            # sum over stores(1) and days(2)
revenue_per_product = units_per_product * prices       # broadcast (4,)*(4,)
for name, rev in zip(products, revenue_per_product):
    print(f"  {name:10s}: ${rev:,.2f}")

best = products[revenue_per_product.argmax()]
print("Top revenue product:", best)


In [ ]:
# 2) Which store sold the most units overall?
units_per_store = sales.sum(axis=(0, 2))               # sum over products(0) and days(2)
print("units per store:", dict(zip(stores, units_per_store)))
print("Best store     :", stores[units_per_store.argmax()])

# 3) Daily total revenue (revenue summed across products & stores per day)
#    sales: (4,5,30); weight each product by its price via broadcasting.
revenue_cube = sales * prices[:, np.newaxis, np.newaxis]   # (4,1,1) broadcasts over (4,5,30)
daily_revenue = revenue_cube.sum(axis=(0, 1))               # -> (30,)
print("\nbest sales day (1-indexed):", daily_revenue.argmax() + 1,
      f"  (${daily_revenue.max():,.2f})")
print("avg daily revenue:", f"${daily_revenue.mean():,.2f}")


In [ ]:
# 4) Boolean analytics: on how many (store, day) combos did 'Widget' sell > 40 units?
widget = sales[0]                       # shape (5, 30)
hot = widget > 40
print("Widget 'hot' selling slots (>40 units):", hot.sum(), "of", widget.size)
print("Stores that ever had a hot Widget day:",
      stores[np.any(hot, axis=1)])


### 📊 Visualize the capstone: dashboard

Three views of the same `sales` cube: revenue ranking, daily revenue trend, and a store×product heatmap.


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

# (a) Revenue per product
order = revenue_per_product.argsort()[::-1]
ax[0].bar(products[order], revenue_per_product[order], color="teal")
ax[0].set_title("Revenue by product"); ax[0].set_ylabel("$")
ax[0].tick_params(axis="x", rotation=20)

# (b) Daily revenue trend with a 7-day moving average
ax[1].plot(np.arange(1, days + 1), daily_revenue, color="gray", alpha=0.6, label="daily")
kernel = np.ones(7) / 7
smooth = np.convolve(daily_revenue, kernel, mode="valid")
ax[1].plot(np.arange(4, days - 2), smooth, color="crimson", lw=2, label="7-day avg")
ax[1].set_title("Daily revenue"); ax[1].set_xlabel("day"); ax[1].legend()

# (c) Heatmap: total units per (store, product)
units_store_product = sales.sum(axis=2).T          # (stores, products)
im = ax[2].imshow(units_store_product, cmap="YlOrRd", aspect="auto")
ax[2].set_xticks(range(len(products))); ax[2].set_xticklabels(products, rotation=20)
ax[2].set_yticks(range(len(stores)));   ax[2].set_yticklabels(stores)
ax[2].set_title("Units sold: store × product")
fig.colorbar(im, ax=ax[2], label="units")

plt.tight_layout(); plt.show()


### 🎯 Final challenges
1. Compute each product's **best single store-day** (max units) and where it happened.
2. Find the product with the **most consistent** daily total sales (lowest standard deviation of its daily totals).
3. Apply a 10% price increase to the two lowest-revenue products and recompute total revenue.


## Part 16 — Bonus: animations 🎬

Animations turn a *sequence of NumPy arrays* into motion — perfect for watching a process unfold (a wave traveling, an estimate converging, an algorithm iterating).

**How it works:** `matplotlib.animation.FuncAnimation` calls an `update(frame)` function once per frame. Each frame you recompute arrays and update the plot. We render with **`anim.to_jshtml()`**, which embeds a JavaScript player right in the notebook — **no ffmpeg required**, and it has play/pause/scrub controls.

> Tip: keep frame count modest (≤ ~150) so the embedded HTML stays light. If a static picture appears instead of a player, make sure the setup cell ran (`%matplotlib inline`).


In [ ]:
from matplotlib import animation
from IPython.display import HTML

# Animation 1: a traveling sine wave. Each frame shifts the phase.
x = np.linspace(0, 2 * np.pi, 400)

fig, ax = plt.subplots(figsize=(7, 3.5))
(line,) = ax.plot(x, np.sin(x), color="royalblue", lw=2)
ax.set_ylim(-1.3, 1.3)
ax.set_title("Traveling wave: sin(x − phase)")
ax.set_xlabel("x")

def update(frame):
    phase = frame / 20 * 2 * np.pi      # full cycle over the frames
    line.set_ydata(np.sin(x - phase))   # recompute the whole array per frame
    return (line,)

anim = animation.FuncAnimation(fig, update, frames=60, interval=50, blit=True)
plt.close(fig)                          # prevents a duplicate static figure
HTML(anim.to_jshtml())


### Watch an estimate converge

A great use of animation: showing an algorithm get better over time. Here we replay the **Monte Carlo π** estimate — as we add random points, the running estimate (top) wobbles toward the true value, while the scatter (bottom) fills the square.


In [ ]:
rng = np.random.default_rng(0)
total = 3000
pts = rng.random((total, 2))
inside = (pts[:, 0]**2 + pts[:, 1]**2) <= 1.0

# Precompute the running estimate after each point (vectorized cumulative mean).
running_pi = 4 * np.cumsum(inside) / np.arange(1, total + 1)

frames = 60
counts = np.linspace(50, total, frames).astype(int)   # points shown per frame

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(6, 7), gridspec_kw={"height_ratios": [1, 2]})

# Top: convergence curve
ax_top.axhline(np.pi, color="black", ls="--", lw=1, label="true π")
(curve,) = ax_top.plot([], [], color="crimson", lw=1.5)
ax_top.set_xlim(0, total); ax_top.set_ylim(2.4, 4.0)
ax_top.set_title("Running π estimate"); ax_top.set_xlabel("points"); ax_top.legend(loc="upper right")

# Bottom: the scatter inside/outside the quarter circle
theta = np.linspace(0, np.pi / 2, 100)
ax_bot.plot(np.cos(theta), np.sin(theta), "k-", lw=2)
sc_in  = ax_bot.scatter([], [], s=6, color="seagreen")
sc_out = ax_bot.scatter([], [], s=6, color="lightcoral")
ax_bot.set_xlim(0, 1); ax_bot.set_ylim(0, 1); ax_bot.set_aspect("equal")
txt = ax_bot.text(0.05, 0.9, "", fontsize=12, weight="bold")

def update(i):
    n = counts[i]
    curve.set_data(np.arange(1, n + 1), running_pi[:n])
    p, ins = pts[:n], inside[:n]
    sc_in.set_offsets(p[ins])
    sc_out.set_offsets(p[~ins])
    txt.set_text(f"n={n}\nπ≈{running_pi[n-1]:.4f}")
    return curve, sc_in, sc_out, txt

anim = animation.FuncAnimation(fig, update, frames=frames, interval=120, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())


**Saving an animation to a file (optional):**
```python
anim.save("wave.gif", writer="pillow", fps=20)   # GIF — needs the 'pillow' package
anim.save("wave.mp4", writer="ffmpeg", fps=20)   # MP4 — needs ffmpeg installed
```
The in-notebook `to_jshtml()` player needs neither — it's the easiest path.

### 🎯 Try it yourself
1. Animate `y = sin(x) * exp(-x/5)` (a damped wave) by shifting the phase.
2. Animate a dot moving around a circle: `x=cos(t), y=sin(t)` for `t` increasing each frame.


## Part 17 — Deep dive: the Fast Fourier Transform (FFT) 🌊

The **FFT** decomposes a signal into the **frequencies** it's made of. It's everywhere: audio/music analysis, image compression (JPEG), radio, vibration monitoring, denoising. NumPy's `np.fft` makes it a one-liner.

Key functions:
- **`np.fft.rfft(y)`** — FFT of a *real* signal; returns complex coefficients (one per frequency). The `r` version drops redundant negative frequencies.
- **`np.fft.rfftfreq(n, d=dt)`** — the actual frequency (in Hz) for each coefficient, given sample spacing `dt`.
- **`np.abs(coeffs)`** — magnitude = "how much" of each frequency is present.
- **`np.fft.irfft`** — inverse: turn coefficients back into a signal.

The golden rule: a tall spike in the spectrum at frequency *f* means your signal contains a sine wave at *f* Hz.


In [ ]:
# Build a signal: 5 Hz + 12 Hz sine waves + random noise.
fs = 500                      # sampling rate (samples per second)
T = 2.0                       # seconds
t = np.linspace(0, T, int(fs * T), endpoint=False)

clean = 1.0 * np.sin(2 * np.pi * 5 * t) + 0.5 * np.sin(2 * np.pi * 12 * t)
rng = np.random.default_rng(0)
signal = clean + 0.8 * rng.standard_normal(t.size)   # add noise

# --- FFT ---
coeffs = np.fft.rfft(signal)
freqs  = np.fft.rfftfreq(t.size, d=1 / fs)
mag    = np.abs(coeffs) / t.size * 2     # normalize to amplitude

# The two biggest peaks should sit at ~5 Hz and ~12 Hz:
peak_idx = np.argsort(mag)[::-1][:2]
print("Detected dominant frequencies (Hz):", np.sort(freqs[peak_idx]).round(2))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

ax[0].plot(t[:200], signal[:200], color="gray", lw=1, label="noisy")
ax[0].plot(t[:200], clean[:200], color="crimson", lw=1.5, label="true")
ax[0].set_title("Signal (first 0.4 s)"); ax[0].set_xlabel("time (s)"); ax[0].legend()

ax[1].plot(freqs, mag, color="navy")
ax[1].set_xlim(0, 30)
ax[1].set_title("Frequency spectrum (FFT)")
ax[1].set_xlabel("frequency (Hz)"); ax[1].set_ylabel("amplitude")
for f in (5, 12):
    ax[1].axvline(f, color="orange", ls="--", alpha=0.7)
plt.tight_layout(); plt.show()


### Denoising with the FFT

Because noise is spread thinly across *all* frequencies while the signal concentrates in a few tall spikes, we can **zero out the small coefficients** and invert — a crude but effective low-pass/threshold filter.


In [ ]:
# Keep only strong frequency components, drop the rest, then invert.
threshold = mag.max() * 0.2
filtered_coeffs = coeffs.copy()
filtered_coeffs[mag < threshold] = 0
recovered = np.fft.irfft(filtered_coeffs, n=t.size)

plt.figure()
plt.plot(t[:200], signal[:200],   color="lightgray", label="noisy input")
plt.plot(t[:200], clean[:200],    color="crimson", lw=2, label="true signal")
plt.plot(t[:200], recovered[:200], color="green", ls="--", lw=1.5, label="FFT-denoised")
plt.title("Denoising by thresholding FFT coefficients")
plt.xlabel("time (s)"); plt.legend()
plt.show()

rmse = np.sqrt(np.mean((recovered - clean) ** 2))
print(f"RMSE vs true signal: {rmse:.3f}  (noise std was 0.8)")


### 🎯 Try it yourself
1. Add a 3rd sine wave at 40 Hz and confirm a new spike appears in the spectrum.
2. Use `np.fft.fft2` on a 2D array (an image) and view `np.abs(np.fft.fftshift(...))` — the 2D spectrum.
3. Lower the threshold to `0.05 * mag.max()` — does more noise leak back in?


## Part 18 — Deep dive: masked arrays (`np.ma`) for messy data 🩹

Real datasets have **missing, invalid, or flagged** values: failed sensor reads, `-999` sentinels, out-of-range outliers. A **masked array** pairs your data with a boolean **mask** marking which entries to ignore. Reductions then **skip masked entries automatically** — cleaner than juggling `nan` everywhere.

`np.ma` vs `nan`:
- `nan` only works for floats and silently poisons `mean`/`sum` (you must remember `nanmean`).
- Masked arrays work for **any dtype** (ints, strings), let you mask by *any condition*, and every operation respects the mask.


In [ ]:
raw = np.array([23.1, 22.8, -999.0, 24.0, 23.5, -999.0, 25.1])   # -999 = sensor failure

# Mask the sentinel values:
data = np.ma.masked_equal(raw, -999.0)
print("masked array :", data)
print("mask         :", data.mask)
print("count valid  :", data.count())

# Reductions ignore masked entries — no nan gymnastics needed:
print("mean (valid) :", data.mean().round(3))
print("max  (valid) :", data.max())
print("std  (valid) :", data.std().round(3))


In [ ]:
# Mask by a condition, not just a sentinel. Mask outliers outside [0, 100].
readings = np.array([12, 45, 300, 67, -5, 88, 23])
clean = np.ma.masked_outside(readings, 0, 100)
print("kept         :", clean)
print("mean of valid:", clean.mean())

# Fill masked entries for export (e.g. back to a plain array):
print("filled w/ -1 :", clean.filled(-1))

# Masks combine through math: invalid stays invalid.
a = np.ma.array([1, 2, 3, 4], mask=[0, 1, 0, 0])
b = np.ma.array([10, 20, 30, 40], mask=[0, 0, 1, 0])
print("a + b        :", a + b, "  (masked where EITHER was masked)")


### 📊 Visualize: gaps are skipped, not plotted as garbage

matplotlib understands masked arrays — masked points create gaps in the line instead of dropping to a sentinel value or `nan` artifacts.


In [ ]:
t = np.linspace(0, 10, 100)
y = np.sin(t) + 0.1 * np.random.default_rng(1).standard_normal(100)
# Pretend the sensor dropped out between t=4 and t=6:
y_masked = np.ma.masked_where((t > 4) & (t < 6), y)

plt.figure()
plt.plot(t, y_masked, "o-", ms=3, color="teal")
plt.title("Masked region (t∈[4,6]) is skipped, not faked")
plt.xlabel("t"); plt.ylabel("signal")
plt.show()
print("valid points plotted:", y_masked.count(), "of", y_masked.size)


### 🎯 Try it yourself
1. Given `temps = np.ma.masked_equal([20, -999, 22, -999, 25], -999)`, compute the mean and how many readings were valid.
2. Mask all negative values in `np.array([3, -1, 4, -2, 5])` with `np.ma.masked_less(a, 0)` and sum the rest.
3. Compare `np.ma.masked_invalid([1, np.nan, 3]).mean()` with `np.nanmean([1, np.nan, 3])`.


## Part 19 — Deep dive: memory-mapped files for data bigger than RAM 💾

What if your array is **larger than your computer's memory**? A **memory-map** (`np.memmap`) maps a file on disk to an array. You index it like a normal array, but the OS loads **only the pages you actually touch** — so you can work with 100 GB of data using a few MB of RAM.

Use cases: huge scientific datasets, embeddings, raw sensor logs, video frames, ML feature stores.

Two doorways:
- **`np.memmap(path, dtype, mode, shape)`** — low-level, full control, raw binary file.
- **`np.load(path, mmap_mode='r')`** — lazily memory-map a `.npy` file saved with `np.save`. Easiest in practice.


In [ ]:
import os, tempfile

path = os.path.join(tempfile.gettempdir(), "numpy_demo.dat")
shape = (10_000, 100)        # 1,000,000 float64 ≈ 8 MB on disk (pretend it's 800 GB)

# --- Create & write, page by page, without holding it all in RAM ---
mm = np.memmap(path, dtype="float64", mode="w+", shape=shape)
for i in range(0, shape[0], 1000):           # fill 1000 rows at a time
    mm[i:i + 1000] = np.arange(i, i + 1000)[:, None] + np.arange(100)
mm.flush()                                   # ensure it's written to disk
print("file size on disk:", os.path.getsize(path) // 1024, "KB")
del mm                                        # close the map


In [ ]:
# --- Reopen read-only and access ONLY what we need ---
mm = np.memmap(path, dtype="float64", mode="r", shape=shape)
print("type            :", type(mm).__name__)
print("row 5000, col 7 :", mm[5000, 7])       # only this page is read from disk
print("mean of 1 column:", mm[:, 0].mean())   # streams just that column
print("a slice is a view, not a 8MB copy:", mm[100:103, :3])
del mm


In [ ]:
# --- The easy path: np.save + lazy np.load(mmap_mode='r') ---
npy_path = os.path.join(tempfile.gettempdir(), "numpy_demo.npy")
big = np.arange(1_000_000, dtype="int32").reshape(1000, 1000)
np.save(npy_path, big)

lazy = np.load(npy_path, mmap_mode="r")        # nothing loaded yet
print("lazy object   :", type(lazy).__name__, lazy.shape, lazy.dtype)
print("touch one row :", lazy[500, :5])        # only this row is paged in
print("reduce a slice:", lazy[:10].sum())      # only 10 rows read

# Cleanup the demo files
del lazy
for p in (path, npy_path):
    try: os.remove(p)
    except OSError: pass
print("demo files cleaned up")


**When to reach for memmap**
- The array doesn't fit in RAM, or you only need slices of a big file.
- You want fast restart/resume — data persists on disk between runs.
- Multiple processes need to share the same large read-only array.

**Watch out for:**
- `mode="w+"` **creates/overwrites**; use `"r"` (read-only) or `"r+"` (read/write existing) to avoid wiping data.
- Call `.flush()` before relying on writes hitting disk.
- Random access across a huge file can be slow (disk seeks) — sequential access is fastest.

### 🎯 Try it yourself
1. Create a `(5000, 5000)` float32 memmap (~100 MB), write `row[i] = i`, flush, reopen read-only, and verify `mm[4999, 0]`.
2. Save an array with `np.save`, reopen with `mmap_mode='r'`, and time reading one row vs `np.load` (full load).

---

## 🎓 You made it!

You now know NumPy from array creation through broadcasting, the memory model, `einsum`, and performance engineering.

### Where to go next
- **Practice**: re-solve the 🎯 exercises without looking; that's where it sticks.
- **pandas** — labeled tabular data, built on NumPy.
- **matplotlib** — plot your arrays.
- **SciPy** — optimization, signal processing, stats.
- **scikit-learn / PyTorch** — ML, all NumPy-flavored.

### Cheat-sheet to remember
- *See a number loop?* → vectorize.
- *Different shapes?* → think broadcasting (compare shapes right-to-left).
- *The named axis disappears* in a reduction.
- *Slice = view, fancy/boolean index = copy.* When unsure, `.copy()`.
- *`@` is matrix mult, `*` is elementwise.*
- *Use `&`/`|` with parentheses, never `and`/`or`.*

Happy array-wrangling! 🧮✨
